In [ ]:
# =========================
# nb_aggregate
# =========================

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

from pyspark.sql import functions as F

# ---------- Stage 2 input ----------
stage2 = spark.table("individual_dfm_consolidated").filter(
    (F.col("period") == period) & (F.col("run_id") == run_id)
)

if stage2.rdd.isEmpty():
    print(f"[nb_aggregate] No Stage 2 rows for period={period}, run_id={run_id}")
    mssparkutils.notebook.exit("NO_DATA")

# Only publish rows marked Include.
publishable = stage2.filter(F.lower(F.col("include_flag")) == "include")
if publishable.rdd.isEmpty():
    print("[nb_aggregate] No Include rows available for Stage 3")
    mssparkutils.notebook.exit("NO_DATA")

# ---------- Stage 3 aggregate ----------
agg = (
    publishable.groupBy(
        "period",
        "run_id",
        "dfm_id",
        "policyholder_number",
        "security_code",
        "isin",
        "sedol",
        "asset_name"
    )
    .agg(
        F.sum(F.coalesce(F.col("holding").cast("double"), F.lit(0.0))).alias("holding"),
        F.sum(F.coalesce(F.col("cash_value_gbp").cast("double"), F.lit(0.0))).alias("cash_value_gbp"),
        F.sum(F.coalesce(F.col("bid_value_gbp").cast("double"), F.lit(0.0))).alias("bid_value_gbp"),
        F.sum(F.coalesce(F.col("accrued_interest_gbp").cast("double"), F.lit(0.0))).alias("accrued_interest_gbp"),
        F.count("*").cast("int").alias("source_count")
    )
    .withColumn("published_at", F.current_timestamp())
    .select(
        "period",
        "run_id",
        "dfm_id",
        "policyholder_number",
        "security_code",
        "isin",
        "sedol",
        "asset_name",
        F.col("holding").cast("decimal(28,8)").alias("holding"),
        F.col("cash_value_gbp").cast("decimal(28,8)").alias("cash_value_gbp"),
        F.col("bid_value_gbp").cast("decimal(28,8)").alias("bid_value_gbp"),
        F.col("accrued_interest_gbp").cast("decimal(28,8)").alias("accrued_interest_gbp"),
        "source_count",
        "published_at"
    )
)

agg.write.mode("append").saveAsTable("aggregated_dfms_consolidated")

# ---------- Compatibility outputs ----------
policy_agg = (
    publishable.groupBy("period", "run_id", "dfm_id", "policyholder_number")
    .agg(
        F.sum(F.coalesce(F.col("cash_value_gbp").cast("double"), F.lit(0.0))).alias("total_cash_value_gbp"),
        F.sum(F.coalesce(F.col("bid_value_gbp").cast("double"), F.lit(0.0))).alias("total_bid_value_gbp"),
        F.sum(F.coalesce(F.col("accrued_interest_gbp").cast("double"), F.lit(0.0))).alias("total_accrued_interest_gbp")
    )
    .select(
        "period",
        "run_id",
        "dfm_id",
        F.col("policyholder_number").alias("policy_id"),
        F.col("total_cash_value_gbp").cast("decimal(28,10)").alias("total_cash_value_gbp"),
        F.col("total_bid_value_gbp").cast("decimal(28,10)").alias("total_bid_value_gbp"),
        F.col("total_accrued_interest_gbp").cast("decimal(28,10)").alias("total_accrued_interest_gbp")
    )
)
policy_agg.write.mode("append").saveAsTable("policy_aggregates")

tpir = (
    publishable.select(
        F.col("policyholder_number").alias("Policyholder_Number"),
        F.col("security_code").alias("Security_Code"),
        F.col("isin").alias("ISIN"),
        F.col("other_security_id").alias("Other_Security_ID"),
        F.col("id_type").alias("ID_Type"),
        F.col("asset_name").alias("Asset_Name"),
        F.lit(None).cast("decimal(28,10)").alias("Acq_Cost_in_GBP"),
        F.col("cash_value_gbp").cast("decimal(28,10)").alias("Cash_Value_in_GBP"),
        F.col("bid_value_gbp").cast("decimal(28,10)").alias("Bid_Value_in_GBP"),
        F.col("accrued_interest_gbp").cast("decimal(28,10)").alias("Accrued_Interest"),
        F.col("holding").cast("decimal(28,10)").alias("Holding"),
        F.col("local_bid_price").cast("decimal(28,10)").alias("Loc_Bid_Price"),
        F.col("local_currency").alias("Currency_Local")
    )
)
tpir.write.mode("append").saveAsTable("tpir_load_equivalent")

print("[nb_aggregate] Wrote aggregated_dfms_consolidated + compatibility outputs")
mssparkutils.notebook.exit("OK")